In [1]:
import pandas as pd
import numpy as np

In [2]:
# STEP 1: Combine the regional files
files = [
    "barwon_sw_vic_australia_listings.csv",
    "sunshine_coast_qld_australia_listings.csv",
    "western_australia_wa_australia_listings.csv",
    "mornington_peninsula_vic_australia_listings.csv",
    "melbourne_vic_australia_listings.csv",
    "northern_rivers_nsw_australia_listings.csv",
    "sydney_nsw_australia_listings.csv",
    "brisbane_qld_australia_listings.csv",
    "tasmania_tas_australia_listings.csv",
    "mid_north_coast_nsw_australia_listings.csv",
    "barossa_valley_sa_australia_listings.csv",
]
 
dfs = []
for f in files:
    temp = pd.read_csv(f, dtype={"id": str, "host_id": str})
    temp["source_file"] = f
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
print(f"Combined {len(files)} files into {len(df)} total rows.")

Combined 11 files into 94945 total rows.


In [3]:
# step 2 check blanks and duplicates (to be flagged and reported)
#missing values report 
missing_report = df.isna().sum()
missing_report = missing_report[missing_report>0]
print("\nMising value counts by column:")
print(missing_report)

#missing price - DO NOT fill with zero
missing_price_count = df["price"].isna().sum()
print(f"\nRows with missing price: {missing_price_count}"
      f"({missing_price_count / len(df):.2%} of total)")

# Rows with missing price cannot have estimated_sales calculated
# (price * booked_days is undefined if price is null), so they are
# excluded from the fact table rather than assigned a false value of 0,
# which would corrupt every SUM/AVG calculation downstream.
missing_price_rows = df[df["price"].isna()].copy()
df = df[df["price"].notna()].copy()
print(f"Rows remaining after removing missing-price rows: {len(df)}")
 
#Duplicate check
# Exact full-row duplicates (true copy-paste duplicates within/across files)
exact_dupes = df.duplicated(keep="first").sum()
print(f"\nExact duplicate rows found: {exact_dupes}")
df = df.drop_duplicates(keep="first")
print(f"Rows remaining after dropping exact duplicates: {len(df)}")
 
# Note: duplicate 'id' values ACROSS regional files are expected and are
# NOT treated as duplicate rows, each regional export has its own id sequence
# an id collision across states is a coincidence, not a true duplicate listing hence the use of surrogate key 
id_collisions = df["id"].duplicated(keep=False).sum()
print(f"Rows sharing an 'id' value with another row (across files): {id_collisions}")
 


Mising value counts by column:
host_name                  2
neighbourhood_group    88252
price                  13095
last_review            14054
reviews_per_month      14054
license                63846
dtype: int64

Rows with missing price: 13095(13.79% of total)
Rows remaining after removing missing-price rows: 81850

Exact duplicate rows found: 0
Rows remaining after dropping exact duplicates: 81850
Rows sharing an 'id' value with another row (across files): 30325


In [4]:
# STEP 3: Derived measures
df["estimated_booked_days"] = 365 - df["availability_365"]
df["estimated_sales"] = df["price"] * df["estimated_booked_days"]
 
# Minimum nights band: 1, 2-3, 4-7, 8-14, 15-30, 31+
bins = [0, 1, 3, 7, 14, 30, np.inf]
labels = ["1", "2-3", "4-7", "8-14", "15-30", "31+"]
df["minimum_nights_band"] = pd.cut(
    df["minimum_nights"], bins=bins, labels=labels, right=True
)

In [5]:
#sanity check and verify values
df[["price", "estimated_booked_days", "estimated_sales", "minimum_nights_band"]].head()

,price,estimated_booked_days,estimated_sales,minimum_nights_band
0,800.0,0,0.0,2-3
1,500.0,278,139000.0,4-7
2,277.0,45,12465.0,1
3,470.0,109,51230.0,2-3
4,600.0,20,12000.0,2-3


In [6]:
# STEP 4: Build dimension tables (each with a surrogate key)
 
# LOCATION Dimenion
LOCATION_Dimension = (
    df[["state", "state_abrv", "region", "neighbourhood"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
LOCATION_Dimension.insert(0, "LocationKey", range(1, len(LOCATION_Dimension) + 1))
 
# ROOMTYPE Dimension
ROOMTYPE_Dimension = (
    df[["room_type"]].drop_duplicates().reset_index(drop=True)
)
ROOMTYPE_Dimension.insert(0, "RoomTypeKey", range(1, len(ROOMTYPE_Dimension) + 1))

#HOST Dimension
HOST_Dimension = (
    df[["host_id", "host_name"]]
    .sort_values("host_name")  # pushes real names before NaN when grouping
    .groupby("host_id", as_index=False)
    .first()
)
HOST_Dimension["host_id"] = HOST_Dimension["host_id"].astype(str)  # keep source id as text
HOST_Dimension.insert(0, "HostKey", range(1, len(HOST_Dimension) + 1))
 
# MINNIGHTS Dimension
MINNIGHTS_Dimension = pd.DataFrame({"band_label": labels})
MINNIGHTS_Dimension.insert(0, "MinNightsBandKey", range(1, len(MINNIGHTS_Dimension) + 1))

In [7]:
#sanity check
print("LOCATION_Dimension:", LOCATION_Dimension.shape)
print("ROOMTYPE_Dimension", ROOMTYPE_Dimension.shape)
print("HOST_Dimension:", HOST_Dimension.shape)
print("MINNIGHTS_Dimension:", MINNIGHTS_Dimension.shape)

LOCATION_Dimension: (432, 5)
ROOMTYPE_Dimension (4, 2)
HOST_Dimension: (41264, 3)
MINNIGHTS_Dimension: (6, 2)


In [8]:
# STEP 4b: Tourism dimension (external dataset)
TOURISM_Dimension = pd.read_csv("DimTourism_external_dataset.csv", encoding="latin1")
TOURISM_Dimension = TOURISM_Dimension[["state_abrv", "tourism_consumption_billion", "year"]].drop_duplicates().reset_index(drop=True)
TOURISM_Dimension.insert(0, "TouristKey", range(1, len(TOURISM_Dimension) + 1))

print("TOURISM_Dimension:", TOURISM_Dimension.shape)
TOURISM_Dimension

TOURISM_Dimension: (6, 4)


,TouristKey,state_abrv,tourism_consumption_billion,year
0,1,NSW,65.92,2024-25
1,2,VIC,51.40,2024-25
2,3,QLD,49.00,2024-25
3,4,WA,21.09,2024-25
4,5,TAS,5.40,2024-25
5,6,SA,12.00,2024-25


In [9]:
# STEP 5: Build the fact table
df["source_listing_id"] = df["id"].astype(str)
df["host_id_str"] = df["host_id"].astype(str)

fact = df.merge(LOCATION_Dimension, on=["state", "state_abrv", "region", "neighbourhood"])
fact = fact.merge(ROOMTYPE_Dimension, on="room_type")

# remove "host_name" from the join, keep only host_id
fact = fact.merge(
    HOST_Dimension, left_on="host_id_str", right_on="host_id"
)

fact = fact.merge(
    MINNIGHTS_Dimension, left_on="minimum_nights_band", right_on="band_label"
)
fact = fact.merge(TOURISM_Dimension, on="state_abrv", how="left")

fact.insert(0, "ListingKey", range(1, len(fact) + 1))

fact_final = fact[[
    "ListingKey", "LocationKey", "RoomTypeKey", "HostKey", "MinNightsBandKey", "TouristKey",
    "source_listing_id", "price", "estimated_booked_days", "estimated_sales",
]]

In [10]:
#check
print(fact_final.shape)
fact_final.head()

(81850, 10)


,ListingKey,LocationKey,RoomTypeKey,HostKey,MinNightsBandKey,TouristKey,source_listing_id,price,estimated_booked_days,estimated_sales
0,1,1,1,38763,2,2,21302,800.0,0,0.0
1,2,1,1,2666,3,2,27561,500.0,278,139000.0
2,3,2,1,20879,1,2,70449,277.0,45,12465.0
3,4,3,1,26622,2,2,84516,470.0,109,51230.0
4,5,1,1,39834,2,2,195168,600.0,20,12000.0


In [11]:
# STEP 6: Export CSVs
fact_final.to_csv("LISTINGSALES_FactTable.csv", index=False)
LOCATION_Dimension.to_csv("LOCATION_Dimension.csv", index=False)
ROOMTYPE_Dimension.to_csv("ROOMTYPE_Dimension.csv", index=False)
HOST_Dimension.to_csv("HOST_Dimension.csv", index=False)
MINNIGHTS_Dimension.to_csv("MINNIGHTS_Dimension.csv", index=False)
TOURISM_Dimension.to_csv("TOURISM_Dimension.csv",index=False)
 
print("\nExport complete:")
print(f"  LISTINGSALES_FactTable: {len(fact_final)} rows")
print(f"  LOCATION_Dimension:      {len(LOCATION_Dimension)} rows")
print(f"  ROOMTYPE_Dimension:      {len(ROOMTYPE_Dimension)} rows")
print(f"  HOST_Dimension:          {len(HOST_Dimension)} rows")
print(f"  MINNIGHTS_Dimension: {len(MINNIGHTS_Dimension)} rows")
print(f"  TOURISM Dimension: {len(TOURISM_Dimension)} rows")


Export complete:
  LISTINGSALES_FactTable: 81850 rows
  LOCATION_Dimension:      432 rows
  ROOMTYPE_Dimension:      4 rows
  HOST_Dimension:          41264 rows
  MINNIGHTS_Dimension: 6 rows
  TOURISM Dimension: 6 rows
